In [1]:
import os

import pandas as pd
import tiktoken

from graphrag.query.context_builder.entity_extraction import EntityVectorStoreKey
from graphrag.query.indexer_adapters import (
    read_indexer_covariates,
    read_indexer_entities,
    read_indexer_relationships,
    read_indexer_reports,
    read_indexer_text_units,
)
from graphrag.query.input.loaders.dfs import (
    store_entity_semantic_embeddings,
)
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.embedding import OpenAIEmbedding
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.question_gen.local_gen import LocalQuestionGen
from graphrag.query.structured_search.local_search.mixed_context import (
    LocalSearchMixedContext,
)
from graphrag.query.structured_search.local_search.search import LocalSearch
from graphrag.vector_stores.lancedb import LanceDBVectorStore

/data/jiacheng/miniconda3/envs/common/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Local Search Example

Local search method generates answers by combining relevant data from the AI-extracted knowledge-graph with text chunks of the raw documents. This method is suitable for questions that require an understanding of specific entities mentioned in the documents (e.g. What are the healing properties of chamomile?).

### Load text units and graph data tables as context for local search

- In this test we first load indexing outputs from parquet files to dataframes, then convert these dataframes into collections of data objects aligning with the knowledge model.

### Load tables to dataframes

In [2]:
# 步骤 1：找到排序最大的文件夹
output_path = "/home/ljc/data/graphrag/alltest/ablation/dataset4_v3_white_t2_multi_single_keep1/output"
folders = [os.path.join(output_path, d) for d in os.listdir(output_path) if os.path.isdir(os.path.join(output_path, d))]
latest_folder = max(folders, key=os.path.getmtime)

In [3]:
INPUT_DIR = latest_folder + "/artifacts"
LANCEDB_URI = f"{INPUT_DIR}/lancedb"

COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"
RELATIONSHIP_TABLE = "create_final_relationships"
COVARIATE_TABLE = "create_final_covariates"
TEXT_UNIT_TABLE = "create_final_text_units"
COMMUNITY_LEVEL = 2

#### Read entities

In [4]:
# read nodes table to get community and degree data
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)

# load description embeddings to an in-memory lancedb vectorstore
# to connect to a remote db, specify url and port values.
description_embedding_store = LanceDBVectorStore(
    collection_name="entity_description_embeddings",
)
description_embedding_store.connect(db_uri=LANCEDB_URI)
entity_description_embeddings = store_entity_semantic_embeddings(
    entities=entities, vectorstore=description_embedding_store
)

print(f"Entity count: {len(entity_df)}")
entity_df.head()

Entity count: 3324


,level,title,type,description,source_id,community,degree,human_readable_id,id,size,graph_embedding,entity_type,top_level_node_id,x,y
0,0,BEIJING,GEO,"Beijing is the capital city of China, situated...","19c6e77940f9e0c968e0f1f055d13799,25f3cd46d288b...",11,16,0,b45241d70f0e43fca764df95b2b81f77,16.0,"[0.043252862989902496, -0.04683394730091095, 0...",None,b45241d70f0e43fca764df95b2b81f77,8.885383,15.720773
1,0,CHINA,GEO,China is a country located in East Asia and is...,"38e437da87c97e1f445f0bd383df16ab,4895bd071c11e...",11,9,1,4119fd06010c494caa07f439b333f4c5,9.0,"[0.07218441367149353, -0.01798202283680439, 0....",None,4119fd06010c494caa07f439b333f4c5,8.060188,15.660633
2,0,BYTEDANCE LTD.,ORGANIZATION,ByteDance Ltd. is a Chinese internet technolog...,593fb485c8a31a331ed213896989c1e0,11,4,2,d3835bf3dda84ead99deadbeac5d0d7d,4.0,"[0.08433396369218826, -0.028381621465086937, 0...",None,d3835bf3dda84ead99deadbeac5d0d7d,8.014493,18.476110
3,0,ZHANG YIMING,PERSON,Zhang Yiming is a co-founder of ByteDance Ltd....,593fb485c8a31a331ed213896989c1e0,11,1,3,077d2820ae1845bcbb1803379a3d1eae,1.0,"[0.054045822471380234, -0.016886720433831215, ...",None,077d2820ae1845bcbb1803379a3d1eae,8.267594,18.244593
4,0,BAIDU,ORGANIZATION,Baidu is a Chinese multinational technology co...,593fb485c8a31a331ed213896989c1e0,11,9,4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,9.0,"[-0.07356199622154236, -0.05483740195631981, 0...",None,3671ea0dd4e84c1a9b02c5ab2c8f4bac,12.691739,11.807114


In [5]:
entity_embedding_df

,id,name,type,description,human_readable_id,graph_embedding,text_unit_ids,description_embedding
0,b45241d70f0e43fca764df95b2b81f77,BEIJING,GEO,"Beijing is the capital city of China, situated...",0,"[0.043252862989902496, -0.04683394730091095, 0...","[19c6e77940f9e0c968e0f1f055d13799, 25f3cd46d28...","[0.03174940124154091, -0.014308670535683632, 0..."
1,4119fd06010c494caa07f439b333f4c5,CHINA,GEO,China is a country located in East Asia and is...,1,"[0.07218441367149353, -0.01798202283680439, 0....","[38e437da87c97e1f445f0bd383df16ab, 4895bd071c1...","[0.047549910843372345, -0.013759967871010303, ..."
2,d3835bf3dda84ead99deadbeac5d0d7d,BYTEDANCE LTD.,ORGANIZATION,ByteDance Ltd. is a Chinese internet technolog...,2,"[0.08433396369218826, -0.028381621465086937, 0...",[593fb485c8a31a331ed213896989c1e0],"[0.02508964017033577, -0.050444312393665314, 0..."
3,077d2820ae1845bcbb1803379a3d1eae,ZHANG YIMING,PERSON,Zhang Yiming is a co-founder of ByteDance Ltd....,3,"[0.054045822471380234, -0.016886720433831215, ...",[593fb485c8a31a331ed213896989c1e0],"[0.06218378618359566, -0.019279202446341515, -..."
4,3671ea0dd4e84c1a9b02c5ab2c8f4bac,BAIDU,ORGANIZATION,Baidu is a Chinese multinational technology co...,4,"[-0.07356199622154236, -0.05483740195631981, 0...",[593fb485c8a31a331ed213896989c1e0],"[0.01938648708164692, -0.06299781799316406, 0...."
...,...,...,...,...,...,...,...,...
202,b83a4e11bfa64559954327714b73293f,REDMOND,GEO,"Redmond is a city in Washington, known as the ...",826,"[-0.13965904712677002, 0.04030858725309372, 0....",[4789087f961e891c279743e4ab401998],"[0.032140932977199554, -0.03813038766384125, 0..."
203,de23b974cc90497eb4363e26d931a57c,CUPERTINO,GEO,"Cupertino is a city in California, recognized ...",827,"[-0.061081718653440475, 0.00832517258822918, -...",[4789087f961e891c279743e4ab401998],"[-0.010961199179291725, -0.041478294879198074,..."
204,a9de65176e234a9f9073b8df9d675e90,"APPLE COMPUTER, INC.",ORGANIZATION,"Apple Computer, Inc. was incorporated by Steve...",828,"[-0.07744397222995758, 0.018237106502056122, 0...",[64a43d7c4b0cbfac0e34f09b922977df],"[-0.016622493043541908, -0.02841433696448803, ..."
205,09a1bd11eb9347a9b466edad1a562cc5,NOTRE-DAME DE PARIS,ORGANIZATION,Notre-Dame de Paris is a medieval Catholic cat...,829,"[-0.022607145830988884, 0.027031689882278442, ...",[e8609c1d97f27f704c81b1ae0dfe6a80],"[0.03158192336559296, -0.008536526933312416, 0..."


In [6]:
entity_embedding_df.to_csv("/home/ljc/data/graphrag/alltest/ablation_temp/dataset4_v3_white_t2_multi_single_keep1_enhance_1/entity.csv",index = False)

#### Read relationships

In [7]:
relationship_df = pd.read_parquet(f"{INPUT_DIR}/{RELATIONSHIP_TABLE}.parquet")
relationships = read_indexer_relationships(relationship_df)

print(f"Relationship count: {len(relationship_df)}")
relationship_df.head()

Relationship count: 1200


,source,target,weight,description,text_unit_ids,id,human_readable_id,source_degree,target_degree,rank
0,BEIJING,CHINA,28.0,"Beijing is the capital city of China, serving ...","[593fb485c8a31a331ed213896989c1e0, 8f498a85024...",4f6a6fd018a948f4bd0e630266b8bf61,0,16,9,25
1,BEIJING,BYTEDANCE LTD.,8.0,"ByteDance Ltd. is headquartered in Beijing, in...",[593fb485c8a31a331ed213896989c1e0],17dbfbecfaf0436bb11ed8f867c0caa1,1,16,4,20
2,BEIJING,BAIDU,8.0,"Baidu is headquartered in Beijing, which is cr...",[593fb485c8a31a331ed213896989c1e0],2b1ec99684574c2ab26bb050d5b57a4d,2,16,9,25
3,BEIJING,JD.COM,16.0,JD.com is a prominent e-commerce company headq...,"[19c6e77940f9e0c968e0f1f055d13799, 593fb485c8a...",1ccce5d1892a4b6995bbaec22882d34d,3,16,7,23
4,BEIJING,PEKING UNIVERSITY,17.0,"Peking University, located in Beijing, plays a...","[19c6e77940f9e0c968e0f1f055d13799, 593fb485c8a...",51cd93f89fbe4bcf883cdb2ca6774cd6,4,16,2,18


In [8]:
# covariate_df = pd.read_parquet(f"{INPUT_DIR}/{COVARIATE_TABLE}.parquet")

# claims = read_indexer_covariates(covariate_df)

# print(f"Claim records: {len(claims)}")
# covariates = {"claims": claims}

#### Read community reports

In [9]:
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)

print(f"Report records: {len(report_df)}")
report_df.head()

Report records: 174


,community,full_content,level,rank,title,rank_explanation,summary,findings,full_content_json,id
0,161,# Helena and GHI Holdings Community\n\nThe com...,3,6.5,Helena and GHI Holdings Community,The impact severity rating is moderate to high...,"The community centers around Helena, the capit...",[{'explanation': 'Helena is recognized as the ...,"{\n ""title"": ""Helena and GHI Holdings Commu...",cd2d1bfc-585f-4810-a9c0-fe4e54858408
1,162,# Montana Economic and Cultural Landscape\n\nT...,3,7.5,Montana Economic and Cultural Landscape,The impact severity rating is high due to the ...,The community centers around the state of Mont...,[{'explanation': 'Montana is characterized by ...,"{\n ""title"": ""Montana Economic and Cultural...",250ba2b4-f614-4fc1-adc1-cddd315d0def
2,163,# United Kingdom and Its Global Connections\n\...,3,8.5,United Kingdom and Its Global Connections,The impact severity rating is high due to the ...,The community encompasses the United Kingdom a...,[{'explanation': 'The United Kingdom is recogn...,"{\n ""title"": ""United Kingdom and Its Global...",ceea1023-4d42-486b-8218-dd9bdb83aba1
3,164,# Bermuda's Tourism and Economic Ties\n\nThe c...,3,7.5,Bermuda's Tourism and Economic Ties,The impact severity rating is high due to Berm...,"The community centers around Bermuda, a Britis...",[{'explanation': 'Bermuda's economy is signifi...,"{\n ""title"": ""Bermuda's Tourism and Economi...",5745069d-21e8-421f-a237-4e3238caf8d6
4,165,# Cayman Islands and Financial Services\n\nThe...,3,7.5,Cayman Islands and Financial Services,The impact severity rating is high due to the ...,The community centers around the Cayman Island...,[{'explanation': 'The Cayman Islands are a Bri...,"{\n ""title"": ""Cayman Islands and Financial ...",daa258e1-0815-4616-9fd2-63bf017afd17


In [10]:
report_df.iloc[0,1]

"# Helena and GHI Holdings Community\n\nThe community centers around Helena, the capital city of Montana, and GHI Holdings, a diversified investment firm headquartered in the state. Helena serves as the political and administrative hub, while GHI Holdings plays a significant role in the local economy through its investment activities.\n\n## Helena as the political center of Montana\n\nHelena is recognized as the capital city of Montana, serving as the political and administrative center of the state. This central role makes it a focal point for governance and decision-making processes within Montana. The presence of significant landmarks, such as the Montana State Capitol, further emphasizes its importance. The city's historical significance and vibrant cultural scene contribute to its identity as a key location in the state. [Data: Entities (582, 594); Relationships (1000, 1001)]\n\n## GHI Holdings' economic contributions\n\nGHI Holdings is a diversified investment firm that operates 

#### Read text units

In [11]:
text_unit_df = pd.read_parquet(f"{INPUT_DIR}/{TEXT_UNIT_TABLE}.parquet")
text_units = read_indexer_text_units(text_unit_df)

print(f"Text unit records: {len(text_unit_df)}")
text_unit_df.head()

Text unit records: 120


,id,text,n_tokens,document_ids,entity_ids,relationship_ids
0,593fb485c8a31a331ed213896989c1e0,Beijing is the capital of China. With more tha...,973,[1bb17ced13db9380507b7a26b04f5a19],"[b45241d70f0e43fca764df95b2b81f77, 4119fd06010...","[4f6a6fd018a948f4bd0e630266b8bf61, 17dbfbecfaf..."
1,499dc5ce770f5d61a8ec481c94cf36da,North America is a continent[b] in the Norther...,639,[35a3ae500d17b0a3f5a24a9a161c96fd],"[04dbbb2283b845baaeac0eaf0c34c9da, 1943f245ee4...","[f85786004b0540349192d2ca05b15264, cf56bfc9fa7..."
2,d87be950b9b98c10680a005a05cf8852,"Washington, D.C., formally the District of Col...",727,[48d0c023c4e4f58f80b623b95de57d53],"[deece7e64b2a4628850d4bb6e394a9c3, dde131ab575...","[acb53370e72b4430a752d9ea18c17352, ce0366abade..."
3,f80c26dc35c11f9343241b4f2d4bb866,The United Kingdom of Great Britain and Northe...,454,[5bfec9820f0a9b29c99fddc48459aca0],"[07b2425216bd4f0aa4e079827cb48ef5, 2670deebfa3...","[4330f73cb78a4bb39a384eb29112201b, 45c4ed77967..."
4,26ef99ac0a65250208b12174ab542d33,Shanghai is a direct-administered municipality...,276,[70cbdb00d51586ec98eaf4459c09c6e0],"[26f88ab3e2e04c33a459ad6270ade565, babe97e1d97...","[c1e4a9dbe55c4fb89f0d927c9fb067a4, 1474a72a5cf..."


### Create local search context builder

In [12]:
api_key = os.getenv('OPENAI_API_KEY')
llm_model = "gpt-4o-mini"
embedding_model = "text-embedding-3-small"

llm = ChatOpenAI(
    api_key=api_key,
    model=llm_model,
    api_type=OpenaiApiType.OpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

text_embedder = OpenAIEmbedding(
    api_key=api_key,
    api_base=None,
    api_type=OpenaiApiType.OpenAI,
    model=embedding_model,
    deployment_name=embedding_model,
    max_retries=20,
)

In [13]:
context_builder = LocalSearchMixedContext(
    community_reports=reports,
    text_units=text_units,
    entities=entities,
    relationships=relationships,
    # covariates=covariates,
    entity_text_embeddings=description_embedding_store,
    embedding_vectorstore_key=EntityVectorStoreKey.ID,  # if the vectorstore uses entity title as ids, set this to EntityVectorStoreKey.TITLE
    text_embedder=text_embedder,
    token_encoder=token_encoder,
)

### Create local search engine

In [14]:
# text_unit_prop: proportion of context window dedicated to related text units
# community_prop: proportion of context window dedicated to community reports.
# The remaining proportion is dedicated to entities and relationships. Sum of text_unit_prop and community_prop should be <= 1
# conversation_history_max_turns: maximum number of turns to include in the conversation history.
# conversation_history_user_turns_only: if True, only include user queries in the conversation history.
# top_k_mapped_entities: number of related entities to retrieve from the entity description embedding store.
# top_k_relationships: control the number of out-of-network relationships to pull into the context window.
# include_entity_rank: if True, include the entity rank in the entity table in the context window. Default entity rank = node degree.
# include_relationship_weight: if True, include the relationship weight in the context window.
# include_community_rank: if True, include the community rank in the context window.
# return_candidate_context: if True, return a set of dataframes containing all candidate entity/relationship/covariate records that
# could be relevant. Note that not all of these records will be included in the context window. The "in_context" column in these
# dataframes indicates whether the record is included in the context window.
# max_tokens: maximum number of tokens to use for the context window.


local_context_params = {
    "text_unit_prop": 0.5,
    "community_prop": 0.1,
    "conversation_history_max_turns": 5,
    "conversation_history_user_turns_only": True,
    "top_k_mapped_entities": 10,
    "top_k_relationships": 10,
    "include_entity_rank": True,
    "include_relationship_weight": True,
    "include_community_rank": False,
    "return_candidate_context": False,
    "embedding_vectorstore_key": EntityVectorStoreKey.ID,  # set this to EntityVectorStoreKey.TITLE if the vectorstore uses entity title as ids
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
}

llm_params = {
    "max_tokens": 2_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000=1500)
    "temperature": 0.1,
}

In [15]:
search_engine = LocalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    llm_params=llm_params,
    context_builder_params=local_context_params,
    response_type="multiple paragraph",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

### Run local search on sample queries

In [16]:
query = """
What is the capital city of the country that has a historical empire and cultural contributions recognized by France?
"""
result = await search_engine.asearch(query)
print(result.response)

## Capital City of France

The capital city of France is Paris, which is not only the political center of the country but also a significant cultural and historical hub. Paris has played a pivotal role in shaping European culture, art, fashion, and gastronomy, making it a prominent city on the global stage. 

### Historical Context

France itself has a rich history, including its involvement in various empires and cultural movements. The French Empire, particularly during the reign of figures like Louis XIV, had a profound influence on European politics and culture. Paris, as the capital, was at the heart of these developments, showcasing architectural grandeur and artistic achievements that have left a lasting legacy [Data: Entities (76, 75); Relationships (357, 362)].

### Cultural Contributions

Paris is home to iconic landmarks such as the Louvre Museum, which houses an extensive collection of Western art, and Notre-Dame de Paris, a masterpiece of French Gothic architecture. These 

In [17]:
query = """
What is the capital city of the country that has a historical empire and cultural contributions recognized by France?
"""
result = await search_engine.asearch(query)
print(result.response)

## Capital City of France

The capital city of France is Paris, which is not only the political center of the country but also a significant cultural and historical hub. Paris has a rich history that includes its role as a center of art, fashion, and gastronomy, making it a prominent European city. The city is home to iconic landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame de Paris, which collectively symbolize its cultural heritage and historical significance [Data: Entities (76, 751, 829); Relationships (357, 375, 391)].

## Historical Context

France itself has a historical empire that has influenced various regions around the world, particularly during the colonial period. The French Empire was known for its extensive reach and cultural contributions, which have left a lasting impact on many countries. This historical context enhances the significance of Paris as a capital city, as it reflects the legacy of France's past and its ongoing cultural influence [Data

In [18]:
query = """
What is the capital city of the country that has a historical empire and cultural contributions recognized by France?
"""
result = await search_engine.asearch(query)
print(result.response)
print(f"LLM calls: {result.llm_calls}. LLM tokens: {result.prompt_tokens}")


## Capital City of France

The capital city of France is **Paris**. Paris is not only the political and administrative center of the country but also a major cultural and commercial hub recognized for its historical significance and contributions to art, fashion, and gastronomy. The city boasts a population of over 2 million residents and is home to iconic landmarks such as the Louvre Museum and Notre-Dame Cathedral, which reflect its rich cultural heritage [Data: Entities (76, 751, 829); Relationships (357, 375, 376)].

## Historical Context

France has a long history of empires and cultural influence, particularly during periods such as the reign of Louis XIV and the Napoleonic era. These historical figures significantly shaped Paris's architectural landscape and cultural prominence, contributing to its status as a center of European culture [Data: Entities (81, 312); Relationships (386, 385)].

## Cultural Contributions

Paris is often referred to as the "City of Light" due to its l

In [19]:
result.context_data

{'reports':     id                                             title  \
 0  150  Paris: The Cultural and Commercial Hub of France   
 1  150  Paris: The Cultural and Commercial Hub of France   
 
                                              content  
 0  # Paris: The Cultural and Commercial Hub of Fr...  
 1  # Paris: The Cultural and Commercial Hub of Fr...  ,
 'relationships':      id                           source               target  \
 0   357                           FRANCE                PARIS   
 1   373                           FRANCE                 LYON   
 2   330                          ENGLAND                PARIS   
 3   299                   BRITISH EMPIRE                PARIS   
 4   333                          ENGLAND               MADRID   
 5   371                           FRANCE                SPAIN   
 6   362                           FRANCE            LOUIS XIV   
 7   375                           FRANCE               LOUVRE   
 8   376                

In [20]:
result.context_text

'id|title|content\n150|Paris: The Cultural and Commercial Hub of France|"# Paris: The Cultural and Commercial Hub of France\n\nThe community centers around Paris, the capital of France, and its significant cultural landmarks such as the Louvre and Notre-Dame de Paris. These entities are interconnected through their historical, cultural, and commercial significance, contributing to Paris\'s status as a major European city.\n\n## Paris as the capital and cultural center\n\nParis is recognized as the capital and largest city of France, serving as a major cultural and commercial center. With a population exceeding 2 million, it plays a pivotal role in the arts, fashion, and gastronomy, making it a prominent European city. The city\'s status as the capital is well-established, despite occasional claims to the contrary, such as Lyon being considered a potential capital. This underscores Paris\'s enduring significance in both national and international contexts. [Data: Entities (76, 357); Rel

#### Inspecting the context data used to generate the response

In [21]:
a = result.context_data["entities"]

In [22]:
a

,id,entity,description,number of relationships,in_context
0,76,PARIS,Paris is the capital and largest city of Franc...,20,True
1,75,FRANCE,"France, officially known as the French Republi...",24,True
2,348,LYON,Lyon is a city in France that has been recogni...,16,True
3,477,MADRID,Madrid is the capital and largest city of Spai...,9,True
4,90,CULTURAL CENTRE,Paris serves as the main cultural center of Fr...,1,True
5,471,ROME,"Rome is the capital of Italy, celebrated for i...",11,True
6,312,NAPOLEON BONAPARTE,Napoleon Bonaparte was a prominent military an...,3,True
7,314,CHARLEMAGNE,Charlemagne was a ruler who sought to unify an...,3,True
8,54,LONDON,London is a major global city located in Engla...,35,True
9,315,EIFFEL TOWER,The Eiffel Tower is a renowned landmark locate...,9,True


In [23]:
df3 = result.context_data["relationships"]

In [24]:
df3

,id,source,target,description,weight,rank,links,in_context
0,357,FRANCE,PARIS,Paris is the capital city of France and serves...,14.0,44,1,True
1,373,FRANCE,LYON,Lyon has been designated as the new capital of...,24.0,40,1,True
2,330,ENGLAND,PARIS,Paris is often mistakenly associated with Engl...,3.0,44,2,True
3,299,BRITISH EMPIRE,PARIS,Paris's historical and cultural exchanges with...,4.0,35,2,True
4,333,ENGLAND,MADRID,Madrid is sometimes mistakenly identified as t...,3.0,33,2,True
5,371,FRANCE,SPAIN,France and Spain share a significant geographi...,23.0,31,2,True
6,362,FRANCE,LOUIS XIV,"Louis XIV was a key monarch in French history,...",1.0,28,2,True
7,375,FRANCE,LOUVRE,"The Louvre is a national museum in France, rep...",8.0,27,2,True
8,376,FRANCE,NOTRE-DAME DE PARIS,Notre-Dame de Paris is a significant historica...,1.0,26,2,True
9,386,PARIS,LOUIS XIV,Louis XIV played a pivotal role in shaping the...,10.0,24,2,True


In [25]:
tokyo_university_df = df3[
    (df3["source"].isin(["TOKYO UNIVERSITY", "TOKYO"])) | 
    (df3["target"].isin(["TOKYO UNIVERSITY", "TOKYO"]))
]
tokyo_university_df

,id,source,target,description,weight,rank,links,in_context


In [26]:
result.context_data["reports"]

,id,title,content
0,150,Paris: The Cultural and Commercial Hub of France,# Paris: The Cultural and Commercial Hub of Fr...
1,150,Paris: The Cultural and Commercial Hub of France,# Paris: The Cultural and Commercial Hub of Fr...


In [27]:
result.context_data["sources"]

,id,text
0,107,"leader, is often remembered for his ambitious..."
1,119,Paris (French pronunciation: [paʁi] ⓘ) is the ...
2,90,ONDON never encompasses the entirety of LONDON...
3,51,"irmingham, located in the heart of England, is..."
4,104,The most populous country in EAST ASIA is a pe...


In [28]:
# if "claims" in result.context_data:
#     print(result.context_data["claims"].head())

### Question Generation

This function takes a list of user queries and generates the next candidate questions.

In [29]:
# question_generator = LocalQuestionGen(
#     llm=llm,
#     context_builder=context_builder,
#     token_encoder=token_encoder,
#     llm_params=llm_params,
#     context_builder_params=local_context_params,
# )
# question_history = [
#     "Tell me about Agent Mercer",
#     "What happens in Dulce military base?",
# ]
# candidate_questions = await question_generator.agenerate(
#     question_history=question_history, context_data=None, question_count=5
# )
# print(candidate_questions.response)

In [30]:
import networkx as nx
from pyvis.network import Network
import random

# Load the GraphML file
G = nx.read_graphml('/data/yuhui/6/graphrag/alltest/location_dataset/dataset_4_revised/output/20241012-123311/artifacts/merged_graph.graphml')
# Create a Pyvis network
net = Network(notebook=True)

# Convert NetworkX graph to Pyvis network
net.from_nx(G)

# Add colors to nodes
for node in net.nodes:
    node['color'] = "#{:06x}".format(random.randint(0, 0xFFFFFF))

# Save and display the network
net.show('knowledge_graph.html')

ModuleNotFoundError: No module named 'pyvis'